# 11 — AntiBP2 NTCT15 Features & Updated Ensemble

**Goal**: The referenced paper *AntiBP2* (Lata et al.) achieves strong performance using the "binary pattern" (One-Hot Encoding) of the N-terminus (first 15 residues) and C-terminus (last 15 residues). We will extract these features, format them as a 600-dim binary vector, train an SVM (as they did), and stack it into our Phase 8 ensemble to push the AUC higher.

**Verify gates**:
- Extract 30-mer strings (N15 + C15, zero-padded with 'X' if length < 30) for all sequences
- One-hot encode the position-specific residues (30 positions $\times$ 21 possible characters)
- Evaluate a Logistic Regression and SVM on this feature set with Cluster CV
- Update the stacking meta-model and compare the new ensemble AUC to the old one (0.903)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✓')

## Step 11.1 — Extract N15 & C15 Termini

In [ ]:
train_df = pd.read_csv('../data/processed/train_clusters.csv')
test_df  = pd.read_csv('../data/processed/test_clean.csv')
y_true = train_df['Label'].values
cluster_labels = train_df['Cluster'].values

def extract_ntct15(seq):
    """
    Extracts first 15 (N-term) and last 15 (C-term) residues.
    If the sequence is shorter than 15, we pad with 'X'.
    """
    if len(seq) >= 15:
        n15 = seq[:15]
        c15 = seq[-15:]
    else:
        # Pad with X
        n15 = seq + 'X' * (15 - len(seq))
        c15 = 'X' * (15 - len(seq)) + seq
    return list(n15 + c15) # Return as list of 30 characters

train_ntct15 = np.array([extract_ntct15(seq) for seq in train_df['Sequence']])
test_ntct15  = np.array([extract_ntct15(seq) for seq in test_df['Sequence']])

print(f"Train string array shape: {train_ntct15.shape}")
print("Sample:", ''.join(train_ntct15[0]))

## Step 11.2 — One-Hot Encode (Binary Pattern)

In [ ]:
# Amino acid alphabet + 'X' pad character
alphabet = list("ACDEFGHIKLMNPQRSTVWXY")
categories = [alphabet for _ in range(30)] # 30 positions, each can take any of 21 characters

encoder = OneHotEncoder(categories=categories, sparse_output=False, handle_unknown='ignore')

X_train_antibp = encoder.fit_transform(train_ntct15)
X_test_antibp  = encoder.transform(test_ntct15)

print(f'Train AntiBP Binary Features Shape: {X_train_antibp.shape}') # Should be (N, 30 * 21 = 630)


## Step 11.3 — Train Classification Models (Cluster CV)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

oof_lr_antibp = np.zeros(len(X_train_antibp))
oof_svm_antibp = np.zeros(len(X_train_antibp))
fold_metrics_lr = []
fold_metrics_svm = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_antibp, y_true, groups=cluster_labels)):
    X_tr, y_tr = X_train_antibp[train_idx], y_true[train_idx]
    X_va, y_va = X_train_antibp[val_idx], y_true[val_idx]
    
    # 1. Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42, C=0.5, n_jobs=None)
    lr.fit(X_tr, y_tr)
    preds_lr = lr.predict_proba(X_va)[:, 1]
    oof_lr_antibp[val_idx] = preds_lr
    fold_metrics_lr.append(roc_auc_score(y_va, preds_lr))
    
    # 2. Linear SVM
    svm_base = SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, random_state=42, n_jobs=None)
    svm_cal = CalibratedClassifierCV(estimator=svm_base, method='sigmoid', cv=3)
    svm_cal.fit(X_tr, y_tr)
    preds_svm = svm_cal.predict_proba(X_va)[:, 1]
    oof_svm_antibp[val_idx] = preds_svm
    fold_metrics_svm.append(roc_auc_score(y_va, preds_svm))

print(f"AntiBP2 Binary Pattern (LR) CV AUC:  {np.mean(fold_metrics_lr):.4f} ± {np.std(fold_metrics_lr):.4f}")
print(f"AntiBP2 Binary Pattern (SVM) CV AUC: {np.mean(fold_metrics_svm):.4f} ± {np.std(fold_metrics_svm):.4f}")

oof_df = pd.DataFrame({
    'Sequence': train_df['Sequence'],
    'Label': y_true,
    'antibp2_lr_pred': oof_lr_antibp,
    'antibp2_svm_pred': oof_svm_antibp
})
oof_df.to_csv('../data/processed/oof_antibp2.csv', index=False)
print('Saved AntiBP2 Out-Of-Fold predictions!')

## Step 11.4 — Update the Global Stacking Ensemble

In [ ]:
df_tfidf = pd.read_csv('../data/processed/oof_tfidf.csv')
df_trees = pd.read_csv('../data/processed/oof_tuned_trees.csv')

# Stack the old 4 features + the 2 new AntiBP2 features
X_meta_new = np.column_stack([
    df_tfidf['tfidf_lr_pred'], 
    df_tfidf['tfidf_svm_pred'],
    df_trees['tuned_lgb_pred'],
    df_trees['tuned_xgb_pred'],
    oof_lr_antibp,
    oof_svm_antibp
])

old_stack_preds = pd.read_pickle('../data/processed/final_meta_model.pkl').predict_proba(X_meta_new[:, :4])[:, 1]
print(f'Old Ensemble CV AUC: {roc_auc_score(y_true, old_stack_preds):.4f}')

cv_meta = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
new_stack_preds = np.zeros(len(X_meta_new))
fold_metrics_meta = []

for fold, (train_idx, val_idx) in enumerate(cv_meta.split(X_meta_new, y_true, groups=cluster_labels)):
    X_tr, y_tr = X_meta_new[train_idx], y_true[train_idx]
    X_va, y_va = X_meta_new[val_idx], y_true[val_idx]
    
    meta = LogisticRegression(random_state=42, C=0.1)
    meta.fit(X_tr, y_tr)
    preds = meta.predict_proba(X_va)[:, 1]
    new_stack_preds[val_idx] = preds
    fold_metrics_meta.append(roc_auc_score(y_va, preds))

new_auc = roc_auc_score(y_true, new_stack_preds)
print(f'\n>> NEW Updated Ensemble CV AUC (with AntiBP2 features): {new_auc:.4f} <<')

final_meta = LogisticRegression(random_state=42, C=0.1)
final_meta.fit(X_meta_new, y_true)
feature_names = ['TF-IDF LR', 'TF-IDF SVM', 'Tuned LGBM', 'Tuned XGBoost', 'AntiBP2 LR', 'AntiBP2 SVM']

plt.figure(figsize=(10, 4))
sns.barplot(x=feature_names, y=final_meta.coef_[0], palette='viridis')
plt.title('Updated Meta-Model Weights (Model Importance)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()